In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import norm
import os

os.makedirs("plots", exist_ok=True)

rng = np.random.default_rng(42)

COLORS = ["#E63946", "#2A9D8F", "#457B9D"]
N_SUBJECTS = 3

fontsize_x = 20
include_xlabels = False

# ════════════════════════════════════════════════════════════════════════════
# 1.  Population distribution
# ════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(4.0, 2.8))
x = np.linspace(-4, 6, 400)
pdf = (0.65 * norm.pdf(x, 1.0, 1.1)
     + 0.20 * norm.pdf(x, -1.5, 0.5)
     + 0.15 * norm.pdf(x, 4.0, 0.6))
ax.fill_between(x, pdf, alpha=0.35, color="#6A4C93")
ax.plot(x, pdf, color="#6A4C93", lw=1.8)
if include_xlabels:
    ax.set_xlabel(r"$p(Y\mid X)$", fontsize=fontsize_x)
ax.set_ylabel("density", fontsize=9)
ax.set_yticks([])
ax.tick_params(labelsize=8)
ax.spines[['top','right']].set_visible(False)
fig.tight_layout(pad=0.3)
fig.savefig("plots/pop_distribution.pdf", bbox_inches='tight')
plt.close()

# ════════════════════════════════════════════════════════════════════════════
# 2.  Hidden params – single scalar sample Y_i | X per subject
# ════════════════════════════════════════════════════════════════════════════
subject_params = []
for _ in COLORS:
    u = rng.uniform()
    if u < 0.65:
        mu = rng.normal(1.0, 1.1)
    elif u < 0.85:
        mu = rng.normal(-1.5, 0.5)
    else:
        mu = rng.normal(4.0, 0.6)
    subject_params.append(mu)

# shared x limits for hidden params and posteriors
XLIM = (min(subject_params) - 1.5, max(subject_params) + 1.5)

fig, ax = plt.subplots(figsize=(4.0, 2.8))
for mu, c in zip(subject_params, COLORS):
    yi = rng.normal(mu, 0.4)
    ax.axvline(yi, color=c, lw=2.2, alpha=0.85)
if include_xlabels:
    ax.set_xlabel(r"$Y_i$", fontsize=fontsize_x)
ax.set_xlim(XLIM)
ax.set_yticks([])
ax.tick_params(labelsize=8)
ax.spines[['top','right']].set_visible(False)
fig.tight_layout(pad=0.3)
fig.savefig("plots/hidden_params.pdf", bbox_inches='tight')
plt.close()

# ════════════════════════════════════════════════════════════════════════════
# 3.  Time series – 3 stacked panels
# ════════════════════════════════════════════════════════════════════════════
T = 80
t = np.arange(T)
fig, axes = plt.subplots(N_SUBJECTS, 1, figsize=(4.0, 3.6), sharex=True)
for i, (mu, c, ax) in enumerate(zip(subject_params, COLORS, axes)):
    freq = 0.06 + 0.02*i
    signal = mu*0.4 + 0.6*np.sin(2*np.pi*freq*t) + rng.normal(0, 0.45, T)
    ax.plot(t, signal, color=c, lw=1.0)
    ax.set_yticks([])
    ax.spines[['top','right']].set_visible(False)
    if i < N_SUBJECTS-1:
        ax.tick_params(labelbottom=False)
axes[-1].set_xlabel("time", fontsize=fontsize_x)
axes[-1].tick_params(labelsize=10)
fig.tight_layout(pad=0.3, h_pad=0.4)
fig.savefig("plots/time_series.pdf", bbox_inches='tight')
plt.close()

# ════════════════════════════════════════════════════════════════════════════
# 4.  Reconstructed posteriors  – same x limits as hidden_params
# ════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(4.0, 2.8))
x = np.linspace(XLIM[0], XLIM[1], 400)
for mu, c in zip(subject_params, COLORS):
    post_mu  = mu * 0.9 + 0.1*rng.normal(0, 0.2)
    post_sig = 0.28
    ax.fill_between(x, norm.pdf(x, post_mu, post_sig), alpha=0.4, color=c)
    ax.plot(x, norm.pdf(x, post_mu, post_sig), color=c, lw=1.8)
if include_xlabels:
    ax.set_xlabel(r"$p(Y_i \mid \mathbf{Z}_i)$", fontsize=fontsize_x)
ax.set_xlim(XLIM)
ax.set_yticks([])
ax.tick_params(labelsize=8)
ax.spines[['top','right']].set_visible(False)
fig.tight_layout(pad=0.3)
fig.savefig("plots/posteriors.pdf", bbox_inches='tight')
plt.close()

# ════════════════════════════════════════════════════════════════════════════
# 5.  Posterior predictive  – no individual curves
# ════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(4.0, 2.8))
x = np.linspace(-4, 6, 400)
pp = np.zeros_like(x)
for _ in range(800):
    u = rng.uniform()
    if u < 0.65:
        mu_s = rng.normal(1.0, 1.1)
    elif u < 0.85:
        mu_s = rng.normal(-1.5, 0.5)
    else:
        mu_s = rng.normal(4.0, 0.6)
    pp += norm.pdf(x, mu_s, 0.5)
pp /= 800
ax.fill_between(x, pp, alpha=0.35, color="#F4A261")
ax.plot(x, pp, color="#F4A261", lw=1.8)
if include_xlabels:
    ax.set_xlabel(r"$p(Y \mid \mathbf{Z})$", fontsize=fontsize_x)
ax.set_yticks([])
ax.tick_params(labelsize=8)
ax.spines[['top','right']].set_visible(False)
fig.tight_layout(pad=0.3)
fig.savefig("plots/posterior_predictive.pdf", bbox_inches='tight')
plt.close()


print("All plots saved to plots/")